In [0]:
%run ../00-common/01.environment-config

In [0]:
# SOURCE AND TARGET PATHS

source_path = f"{s3_root_path}/Bronze/Aaisha/races"
target_path = f"{s3_root_path}/Silver/Aaisha/races"

print("Source Path :", source_path)
print("Target Path :", target_path)

In [0]:
races_df = (
    spark.read
        .format("delta")
        .load(source_path)
)

display(races_df)

In [0]:
from pyspark.sql import functions as F

races_clean_df = (
    races_df
        .dropna(how="all")
        .dropDuplicates()
        .filter(F.col("season").isNotNull())
        .filter(F.col("round").isNotNull())
        .filter(F.col("raceName").isNotNull())
        .withColumn("raceName", F.trim(F.col("raceName")))
        .withColumn("url", F.trim(F.col("url")))
        .withColumn("circuitId", F.trim(F.col("circuitId")))
)

In [0]:
races_final_df = races_clean_df.select(
    "season",
    "round",
    "raceName",
    "date",
    "circuitId",
    "url",
    "ingestion_timestamp",
    "source_file"
)

display(races_final_df)

In [0]:
(
    races_final_df
        .write
        .format("delta")
        .mode("overwrite")
        .save(target_path)
)

In [0]:
silver_races_df = (
    spark.read
        .format("delta")
        .load(target_path)
)

display(silver_races_df)